In [2]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
import ipywidgets as widgets
import ipyleaflet
from ipyleaflet import Map, FullScreenControl, Marker, Popup, GeoData
from branca.colormap import linear
from IPython.display import display

In [3]:
html_label = widgets.HTML(
    value='<div style="font-family: sans-serif; color: #333; padding: 8px 12px; background: rgba(255,255,255,0.9); border-radius: 4px; box-shadow: 0 1px 5px rgba(0,0,0,0.15);">Hover on a municipality</div>',
)
html_control = ipyleaflet.WidgetControl(widget=html_label, position='topright')

def get_map(title="Portugal", basemap=ipyleaflet.basemaps.CartoDB.Positron) -> Map:
    # ipyleaflet.basemaps.CartoDB.Positron
    # ipyleaflet.basemaps.OpenStreetMap.Mapnik
    # get a map locked to Portuguese boundary
    portugal_center = (39.3999, -8.2245)
    portugal_bounds = [(36.7, -9.7), (42.3, -6.1)] 

    m = Map(
        center=portugal_center,
        zoom=8,
        min_zoom=6,
        max_bounds=portugal_bounds,
        scroll_wheel_zoom=True,
        basemap=basemap,
    )
    m.add(FullScreenControl())

    # title
    map_title = widgets.HTML(
        value=f'<h2 style="font-family:sans-serif; margin:0; padding:4px; color:#333;">{title}</h2>',
        layout=widgets.Layout(padding='0px 10px')
    )
    title_control = ipyleaflet.WidgetControl(widget=map_title, position='topleft')
    m.add_control(title_control)

    return m

def with_html_label(m:Map) -> Map:
    m.add_control(html_control)
    return m

In [4]:
geojson_url = "../../data/processed_data/portugal-municipalities.geojson"


def get_geojson():
    with open(geojson_url, 'r') as f:
        geo_data = json.load(f)
        # promote district and municipal for key mapping on choropleths and remove null districts
        for i in range(len(geo_data["features"])-1, -1, -1): # there are other ways :)
            feature = geo_data["features"][i]
            if feature["properties"]["Concelho"] is None:
                del geo_data["features"][i]
            else:
                geo_data["features"][i]["id"] = feature["properties"]["Concelho"]
        
        return geo_data

In [5]:
def update_html(feature, **kwargs):
    name = feature['properties']['Concelho']
    html_label.value = f'<div style="font-family: sans-serif; color: #333; padding: 8px 12px; background: rgba(255,255,255,0.9); border-radius: 4px; box-shadow: 0 1px 5px rgba(0,0,0,0.15);">Location: <b>{name}</b></div>'

def state_style_callback(feature):
    return {
        'fillColor': '#4a7c59',   # Elegant sage/slate green (neutral yet distinct)
        'color': '#ffffff',       # Crisp white border lines
        'weight': 1.2,            # Thin, sleek borders
        'opacity': 1.0,           # Full opacity for the white line border
        'fillOpacity': 0.35       # Low opacity lets basemap details peek through beautifully
    }

hover_style = {
    'fillColor': '#2b5037',       # Deepens the green shade on hover
    'color': '#ffffff',           # Keeps the crisp white boundary line
    'weight': 2.0,                # Marginally thickens border to emphasize selection
    'fillOpacity': 0.75           # Brightens up the fill significantly
}

style = {
    'fillColor': '#2b5c8f', # Sleek slate blue
    'fillOpacity': 0.3,     # Translucent fill
    'color': '#ffffff',     # Pure white crisp borders
    'weight': 1.5,          # Mid-thin borders
    'dashArray': '3, 3'     # Elegant dashed state borders
}

def overlay_pt_districts(m:Map):
    geojson_layer = ipyleaflet.GeoJSON(
        data=get_geojson(),
        style_callback=state_style_callback,
        style=style,
        hover_style=hover_style,
    )
    geojson_layer.on_hover(update_html)

    m.add_layer(geojson_layer)

In [6]:

def plot_point_map(df: pd.DataFrame, *, label:str='municipality', metric:str='metric', title="Portugal", html_popup_template:str=None):
    """
        Plot a Point Map on a Portugal Municipality layout.

        Note: The longitude and latitude columns are required in the dataframe.
    """
    m = get_map(title=title)
    m = with_html_label(m)
    overlay_pt_districts(m)

    df = df.copy() # defensive copy
    
    # 4. Iterate over DataFrame to systematically register interactive markers
    for index, row in df.iterrows():
        # construct distinct marker
        marker = Marker(location=(row['latitude'], row['longitude']), draggable=False, opacity=0.6)
        
        # render customized HTML layout popup attached to each landmark coordinate
        message = widgets.HTML(value=f"""
            <div style="font-family: Arial, sans-serif; padding: 5px;">
                <b style="color: #2C3E50; font-size: 14px;">{row[label]}</b><br>
                <span style="color: #7F8C8D;">Performance {metric}:</span> 
                <strong style="color: #27AE60;">{row[metric]}</strong>
            </div>
        """)
        
        marker.popup = message
        m.add(marker)

    display(m)

In [7]:
def plot_choropleth(df: pd.DataFrame, *, label:str="municipality", metric:str="metric", op:str="sum", title="Portugal", html_popup_template:str=None):
    """
        Plot a Choropleth Map on a Portugal Municipality layout.

        Note: The `municipality` column is required in the dataframe.
    """
    m = get_map(title=title)
    m = with_html_label(m)

    df = df.copy() # defensive copy

    agg_df = (
        df.groupby(label).agg(value=(metric, op)).reset_index()
    )

    # generate an income lookup dictionary for ipyleaflet mapping
    # e.g.: {"municipality name": value}
    value_dict = pd.Series(
        agg_df.value.values, index=agg_df[label]
    ).to_dict()

    geo_data = get_geojson()
    
    # some scaling with a log scale to prevent huge number outliers from drowning out smaller ones
    # value_dict = {k: float(np.log1p(v)) for k, v in value_dict.items()}

    # a dynamic Color Map scale with legend formatting and a linear palette gradient.
    colormap = linear.magma.scale(min(value_dict.values()), max(value_dict.values()))

    layer = ipyleaflet.Choropleth(
        geo_data=geo_data,
        choro_data=value_dict,
        colormap=colormap, #linear.Paired_10,#YlOrRd_04,
        border_color='black',
        style={"fillOpacity": 0.6, "dashArray": "5, 5", "weight": 1, "color": "gray"},
        hover_style={
            "fillOpacity": 0.9,
            "color": "#2c3e50",
            "weight": 2,
        },
    )

    def update_hover_label(**kwargs):
        # ipyleaflet packs the GeoJSON feature inside kwargs
        feature = kwargs.get('feature', {})
        properties = feature.get('properties', {})
        
        if properties:
            name = properties.get('Concelho', 'Unknown')
            value = value_dict.get(name, "N/A")
            html_label.value = f"<b>{name}</b><br>Value: {value}"

    layer.on_hover(update_hover_label)
    
    m.add(layer)

    legend = ipyleaflet.ColormapControl(
        colormap=colormap,
        position="bottomright",
        # orientation="vertical",
        title=f"Log-Scaled {metric}",
    )
    m.add(legend)
    display(m)